## 섹션 0-1: 환경 감지 및 경로 설정

In [ ]:
import os, sys

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/datasets/junseopkim/structure-stability-data/data'
    SRC_DIR  = '/kaggle/input/datasets/junseopkim/structure-stability-src/src'
    OUT_DIR  = '/kaggle/working/outputs'
    IMG_SIZE = 384
    NUM_WORKERS = 2
else:
    DATA_DIR = '../data'
    SRC_DIR  = '../src'
    OUT_DIR  = '../outputs'
    IMG_SIZE = 224
    NUM_WORKERS = 0

sys.path.append(SRC_DIR)

FIG_DIR = f"{OUT_DIR}/../reports/figures" if not IS_KAGGLE else f"{OUT_DIR}/figures"
os.makedirs(f'{OUT_DIR}/models', exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs', exist_ok=True)
os.makedirs(f'{OUT_DIR}/submissions', exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"환경: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"IMG_SIZE: {IMG_SIZE}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

## 섹션 0-2: import 및 seed 고정

In [ ]:
from model import MultiViewNet
from experiment_utils import set_seed, EarlyStopping, run_experiment, run_inference, train_one_epoch, evaluate

import random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, accuracy_score
from dataset import build_datasets, build_test_dataset
from augmentation import get_train_transform, get_val_transform

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 섹션 1-2: 구조 확인

In [ ]:
test_model = MultiViewNet(backbone_name='resnet18', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out = test_model(dummy, dummy)
print(f"출력 shape: {out.shape}")
print(f"총 파라미터: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model, dummy

## 섹션 2: EXP-001 (HIGH 증강)

In [ ]:
config_exp001 = {
    "exp_id": "001", "model_version": "2", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "concat",
    "use_physics": False, "use_physics_features": False, "physics_dim": 6,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "imagenet", "dev_split_ratio": 0.0, "img_size": IMG_SIZE,
    "aug_params": {
        "brightness_p": 0.7,
        "gamma_p": 0.5,
        "hsv_p": 0.5,
        "shift_scale_p": 0.4,
        "perspective_p": 0.4,
        "flip_p": 0.3
    },
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 5, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp001 = run_experiment(config_exp001)

## 섹션 3: EXP-002 (Dev 학습 포함)

In [ ]:
config_exp002 = {**config_exp001, "exp_id": "002", "model_version": "3", "dev_split_ratio": 0.8}
result_exp002 = run_experiment(config_exp002)
print("⚠️ Val 샘플 20개 — EXP-001(Val 100개)과 직접 비교 주의")

## 섹션 4: EXP-004 (Custom 정규화)

In [ ]:
config_exp004 = {**config_exp001, "exp_id": "004", "model_version": "4", "norm_version": "custom"}
result_exp004 = run_experiment(config_exp004)

## 섹션 5: EXP-005 (EfficientNet-B0)

In [ ]:
test_eff = MultiViewNet(backbone_name='efficientnet_b0', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(16, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
_ = test_eff(dummy, dummy)
if torch.cuda.is_available():
    print(f"VRAM 사용량: {torch.cuda.memory_allocated()/(1024**2):.1f} MB")
del test_eff, dummy; torch.cuda.empty_cache()

config_exp005 = {**config_exp001, "exp_id": "005", "model_version": "5",
                 "model_name": "efficientnet", "backbone": "efficientnet_b0"}
result_exp005 = run_experiment(config_exp005)

## 섹션 6: EXP-006 (물리 피처)

In [ ]:
config_exp006 = {**config_exp001, "exp_id": "006", "model_version": "6",
                 "use_physics": True, "use_physics_features": True, "physics_dim": 6}
result_exp006 = run_experiment(config_exp006)

## 섹션 7: EXP-007 (Fusion 구조 개선)

In [ ]:
config_exp007 = {**config_exp001, "exp_id": "007", "model_version": "7", "fusion_mode": "diff_concat"}
result_exp007 = run_experiment(config_exp007)

## 섹션 8: EXP-008 (최적 조합)
Custom 정규화 + diff_concat Fusion + Dev 포함 + 물리 피처 + HIGH 증강 전부 합침
결과: Best Val LogLoss 0.1028 (Epoch 2) — Public 0.16137 / 175위

In [ ]:
config_exp008 = {
    "exp_id": "008", "model_version": "8", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "diff_concat",
    "use_physics": True, "use_physics_features": True, "physics_dim": 6,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "custom", "dev_split_ratio": 0.8, "img_size": IMG_SIZE,
    "aug_params": {
        "brightness_p": 0.7, "gamma_p": 0.5, "hsv_p": 0.5,
        "shift_scale_p": 0.4, "perspective_p": 0.4, "flip_p": 0.3
    },
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 7, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp008 = run_experiment(config_exp008)

## 섹션 9: EXP-009~011 (dev_split_ratio Ablation)
ratio=0.0 / 0.5 / 1.0 비교
결과: ratio=0.5 (Val 0.1037) > ratio=0.0 (0.1880) > ratio=1.0 (0.5896)

In [ ]:
for ratio, exp_id, ver in [(0.0, '009', '9'), (0.5, '010', '10'), (1.0, '011', '11')]:
    cfg = {
        **config_exp008,
        "exp_id": exp_id, "model_version": ver,
        "dev_split_ratio": ratio,
        "epochs": 3 if ratio == 1.0 else 50,
        "early_stopping_patience": 7
    }
    result = run_experiment(cfg)
    print(f"ratio={ratio} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

## 섹션 10: EXP-012 (최적 조합 + ratio=0.5)
EXP-008 조합에서 dev_split_ratio 0.8→0.5로 변경
결과: Best Val LogLoss 0.0608 (Epoch 19) — 현재 최고 성능

In [ ]:
config_exp012 = {
    **config_exp008,
    "exp_id": "012", "model_version": "12",
    "dev_split_ratio": 0.5,
    "early_stopping_patience": 10
}
result_exp012 = run_experiment(config_exp012)

## 섹션 11: 추론 — submission_v2.csv 생성 (EXP-008 기준)
Public LogLoss: 0.16137 / 175위

In [ ]:
preds_v2 = run_inference(
    config=config_exp008,
    model_path=result_exp008['model_path']
)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v2
sub_df['stable_prob']   = 1.0 - preds_v2
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v2.csv", index=False)
print(f"submission_v2.csv 저장 완료")
print(f"prob 범위: {preds_v2.min():.4f} ~ {preds_v2.max():.4f}")

## 섹션 12: 전체 실험 결과 비교표

In [ ]:
results_all = [
    {"exp_id": "baseline", "model": "resnet18", "변경사항": "없음(기준점)",    "best_val_logloss": 1.5369, "best_epoch": 1,  "public": 1.67225},
    {"exp_id": "001",      "model": "resnet18", "변경사항": "HIGH 증강",        "best_val_logloss": 0.5007, "best_epoch": 1,  "public": None},
    {"exp_id": "002",      "model": "resnet18", "변경사항": "Dev 포함(0.8)",    "best_val_logloss": 0.3687, "best_epoch": 3,  "public": None},
    {"exp_id": "004",      "model": "resnet18", "변경사항": "Custom 정규화",     "best_val_logloss": 0.1700, "best_epoch": 11, "public": None},
    {"exp_id": "005",      "model": "efficientnet_b0", "변경사항": "EfficientNet-B0", "best_val_logloss": 0.5081, "best_epoch": 1, "public": None},
    {"exp_id": "006",      "model": "resnet18", "변경사항": "물리 피처",         "best_val_logloss": 0.3438, "best_epoch": 12, "public": None},
    {"exp_id": "007",      "model": "resnet18", "변경사항": "diff_concat Fusion","best_val_logloss": 0.1996, "best_epoch": 7,  "public": None},
    {"exp_id": "008",      "model": "resnet18", "변경사항": "최적 조합",         "best_val_logloss": 0.1028, "best_epoch": 2,  "public": 0.16137},
    {"exp_id": "009",      "model": "resnet18", "변경사항": "ratio=0.0",        "best_val_logloss": 0.1880, "best_epoch": 7,  "public": None},
    {"exp_id": "010",      "model": "resnet18", "변경사항": "ratio=0.5",        "best_val_logloss": 0.1037, "best_epoch": 22, "public": None},
    {"exp_id": "011",      "model": "resnet18", "변경사항": "ratio=1.0",        "best_val_logloss": 0.5896, "best_epoch": 1,  "public": None},
    {"exp_id": "012",      "model": "resnet18", "변경사항": "최적조합+ratio=0.5","best_val_logloss": 0.0608, "best_epoch": 19, "public": None},
]
result_df = pd.DataFrame(results_all).sort_values('best_val_logloss')
print(result_df.to_string(index=False))
result_df.to_csv(f"{OUT_DIR}/logs/experiment_summary.csv", index=False)